# Compile Results Across All Schemes

Consolidates full-cohort and feature-comparison results across:
- **death_met**: Death + 7 metastasis sites
- **icd3**: ICD-10 level 3 codes (pre-treatment)
- **icd3_post**: ICD-10 level 3 codes (post-treatment)
- **phecode**: Phecodes (pre-treatment)
- **phecode_post**: Phecodes (post-treatment)

Outputs a single DataFrame with c-index, mean AUC(t), and per-feature metrics for every event across all schemes.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import icd10
    HAS_ICD10 = True
except ImportError:
    HAS_ICD10 = False
    print('icd10 package not available; ICD descriptions will be codes only')

In [ ]:
# === Paths ===
DATA_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/'
SURV_PATH = os.path.join(DATA_PATH, 'time-to-event_analysis/')
RESULTS_BASE = os.path.join(SURV_PATH, 'results/')

# Output
COMPILED_PATH = os.path.join(RESULTS_BASE, 'compiled_all_schemes/')
os.makedirs(COMPILED_PATH, exist_ok=True)

# Scheme config (mirrors slurm_array_utils.SCHEME_CONFIG)
SCHEMES = {
    'death_met': 'death_met_results',
    'icd3': 'level_3_ICD_results',
    'icd3_post': 'level_3_ICD_post_results',
    'phecode': 'phecode_results',
    'phecode_post': 'phecode_post_results',
}

FEATURE_NAMES = ['stage', 'treatment', 'somatic', 'prs', 'text']

MET_EVENTS = {'brainM', 'boneM', 'adrenalM', 'liverM', 'lungM', 'nodeM', 'peritonealM'}
MET_DESCRIPTIONS = {
    'brainM': 'Brain metastasis', 'boneM': 'Bone metastasis',
    'adrenalM': 'Adrenal metastasis', 'liverM': 'Liver metastasis',
    'lungM': 'Lung metastasis', 'nodeM': 'Lymph node metastasis',
    'peritonealM': 'Peritoneal metastasis',
}

In [ ]:
def get_event_description(event, scheme):
    """Look up a human-readable description for an event code."""
    if event == 'death':
        return 'Death (overall survival)'
    if event == 'vte':
        return 'Venous thromboembolism'
    if event in MET_DESCRIPTIONS:
        return MET_DESCRIPTIONS[event]
    if scheme in ('icd3', 'icd3_post') and HAS_ICD10:
        if icd10.exists(event):
            return icd10.find(event).description
    # For phecodes or unresolved ICD codes, return the code itself
    return event


def get_event_category(event, scheme):
    """Classify event into a broad category for plotting."""
    if event == 'death':
        return 'Death'
    if event in MET_EVENTS:
        return 'Metastasis'
    if event == 'vte':
        return 'VTE'
    category_map = {
        'icd3': 'icd3',
        'icd3_post': 'icd3_post',
        'phecode': 'phecode',
        'phecode_post': 'phecode_post',
    }
    return category_map.get(scheme, scheme)


def get_icd_chapter(event, scheme):
    """Get ICD chapter info if applicable."""
    if scheme in ('icd3', 'icd3_post') and HAS_ICD10 and icd10.exists(event):
        code = icd10.find(event)
        try:
            return code.chapter, code.block_description
        except Exception:
            return code.chapter, None
    return None, None

## Compile full-cohort + feature-comparison metrics across all schemes

In [ ]:
all_rows = []

for scheme, results_dir in SCHEMES.items():
    full_cohort_path = os.path.join(RESULTS_BASE, results_dir, 'full_cohort')
    feature_comps_path = os.path.join(RESULTS_BASE, results_dir, 'feature_comps')

    if not os.path.isdir(full_cohort_path):
        print(f'  Skipping {scheme}: {full_cohort_path} not found')
        continue

    # Find events that have full-cohort results
    events = [e for e in os.listdir(full_cohort_path)
              if os.path.isdir(os.path.join(full_cohort_path, e))]

    print(f'{scheme}: {len(events)} events found')

    for event in events:
        row = {'scheme': scheme, 'event': event}

        # --- Event metadata ---
        row['event_description'] = get_event_description(event, scheme)
        row['event_category'] = get_event_category(event, scheme)
        row['icd_chapter'], row['icd_block_description'] = get_icd_chapter(event, scheme)

        # --- Full cohort: base and text models ---
        event_path = os.path.join(full_cohort_path, event)
        for model_name, file_name in [('base', 'base_test.csv'), ('text', 'text_test.csv')]:
            fpath = os.path.join(event_path, file_name)
            if os.path.isfile(fpath):
                df = pd.read_csv(fpath)
                for metric in ['mean_c_index', 'mean_auc(t)']:
                    if metric in df.columns:
                        row[f'{model_name}_{metric}'] = df[metric].values[0]

        # --- Feature comparisons ---
        feat_event_path = os.path.join(feature_comps_path, event)
        if os.path.isdir(feat_event_path):
            for feature in FEATURE_NAMES:
                feat_file = os.path.join(feat_event_path, f'{feature}_test.csv')
                if os.path.isfile(feat_file):
                    df = pd.read_csv(feat_file)
                    for metric in ['mean_c_index', 'mean_auc(t)']:
                        if metric in df.columns:
                            row[f'{feature}_{metric}'] = df[metric].values[0]

        all_rows.append(row)

results_df = pd.DataFrame(all_rows)
print(f'\nTotal: {len(results_df)} event-scheme rows')
results_df['scheme'].value_counts()

In [ ]:
# Compute improvement from adding text embeddings
results_df['delta_c_index'] = results_df['text_mean_c_index'] - results_df['base_mean_c_index']
results_df['delta_mean_auc'] = results_df['text_mean_auc(t)'] - results_df['base_mean_auc(t)']

# Save
results_df.to_csv(os.path.join(COMPILED_PATH, 'all_schemes_compiled_metrics.csv'), index=False)
print(f'Saved to {os.path.join(COMPILED_PATH, "all_schemes_compiled_metrics.csv")}')
results_df.head()

## Summary statistics by scheme

In [ ]:
summary = (results_df
    .groupby('scheme')
    .agg(
        n_events=('event', 'count'),
        mean_base_c_index=('base_mean_c_index', 'mean'),
        mean_text_c_index=('text_mean_c_index', 'mean'),
        mean_delta_c_index=('delta_c_index', 'mean'),
        median_delta_c_index=('delta_c_index', 'median'),
        pct_improved=('delta_c_index', lambda x: (x > 0).mean() * 100),
    )
    .round(4)
)
summary

## Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for scheme in SCHEMES:
    subset = results_df.loc[results_df['scheme'] == scheme, 'delta_c_index'].dropna()
    if len(subset) > 0:
        ax.hist(subset, bins=30, alpha=0.5, label=f'{scheme} (n={len(subset)}, mean={subset.mean():.3f})')
ax.axvline(0, color='black', linewidth=1)
ax.set_xlabel('Delta C-index (text - base)')
ax.set_ylabel('Count')
ax.set_title('Improvement in C-index with Text Embeddings')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(COMPILED_PATH, 'delta_c_index_by_scheme.png'), dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
for scheme in SCHEMES:
    subset = results_df.loc[results_df['scheme'] == scheme].dropna(subset=['base_mean_c_index', 'text_mean_c_index'])
    ax.scatter(subset['base_mean_c_index'], subset['text_mean_c_index'],
               alpha=0.5, s=20, label=f'{scheme} (n={len(subset)})')
lims = [0.45, 1.0]
ax.plot(lims, lims, 'k--', linewidth=0.8)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('Base model C-index')
ax.set_ylabel('Text model C-index')
ax.set_title('Base vs Text Model C-index Across All Schemes')
ax.set_aspect('equal')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(COMPILED_PATH, 'scatter_base_vs_text_all_schemes.png'), dpi=150)
plt.show()

In [ ]:
from scipy.stats import ttest_rel

category_order = ['Death', 'Metastasis', 'icd3', 'icd3_post', 'phecode', 'phecode_post']
categories_present = [c for c in category_order if c in results_df['event_category'].values]

long = results_df.melt(
    id_vars=['event', 'event_category'],
    value_vars=['base_mean_c_index', 'text_mean_c_index'],
    var_name='model', value_name='c_index'
).dropna()
long['model'] = long['model'].map({'base_mean_c_index': 'Base', 'text_mean_c_index': 'Text'})

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=long, x='event_category', y='c_index', hue='model',
            order=categories_present, errorbar='sd', ax=ax)
ax.set_xlabel('')
ax.set_ylabel('C-index')
ax.set_title('Base vs Text Model Performance by Event Category')
plt.tight_layout()
plt.savefig(os.path.join(COMPILED_PATH, 'bar_base_vs_text_by_category.png'), dpi=150)
plt.show()

In [ ]:
# Feature comparison violin plot (across all schemes)
feature_cols = [f'{f}_mean_c_index' for f in FEATURE_NAMES]
available_feat_cols = [c for c in feature_cols if c in results_df.columns]

if available_feat_cols:
    feat_long = results_df.melt(
        id_vars=['event', 'scheme'],
        value_vars=available_feat_cols,
        var_name='feature', value_name='c_index'
    ).dropna()
    feat_long['feature'] = feat_long['feature'].str.replace('_mean_c_index', '')

    order = feat_long.groupby('feature')['c_index'].mean().sort_values(ascending=False).index.tolist()

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.violinplot(data=feat_long, x='feature', y='c_index', order=order,
                   inner='quartile', cut=0, ax=ax)
    ax.axhline(0.5, color='black', linewidth=0.8)
    ax.set_xlabel('')
    ax.set_ylabel('C-index')
    ax.set_title('Feature Class C-index Distribution (All Schemes)')
    plt.tight_layout()
    plt.savefig(os.path.join(COMPILED_PATH, 'violin_feature_comps_all_schemes.png'), dpi=150)
    plt.show()

## Top events by text improvement

In [ ]:
top_improved = (results_df
    .dropna(subset=['delta_c_index'])
    .sort_values('delta_c_index', ascending=False)
    .head(25)
    [['scheme', 'event', 'event_description', 'event_category',
      'base_mean_c_index', 'text_mean_c_index', 'delta_c_index']]
    .round(3)
)
top_improved

In [ ]:
# Top 5 most improved per scheme
for scheme in SCHEMES:
    subset = results_df.loc[results_df['scheme'] == scheme].dropna(subset=['delta_c_index'])
    top5 = subset.nlargest(5, 'delta_c_index')[['event', 'event_description', 'base_mean_c_index', 'text_mean_c_index', 'delta_c_index']].round(3)
    print(f'\n--- {scheme} (top 5 improved) ---')
    print(top5.to_string(index=False))

In [ ]:
# Death + metastasis summary
death_met = results_df.loc[results_df['scheme'] == 'death_met'].copy()
if not death_met.empty:
    print('Death + Metastasis results:')
    print(death_met[['event', 'event_description', 'base_mean_c_index', 'text_mean_c_index', 'delta_c_index']]
          .sort_values('text_mean_c_index', ascending=False)
          .round(3)
          .to_string(index=False))